In [32]:
# Linguistic Analysis of English Pop Love Song Lyrics (spaCy)

# - Dataset: 100 English pop love song lyric text files in ./lyrics
# - Tool: spaCy (en_core_web_md)
# - Outputs:
#   1) Example file analysis + OOV error candidate summary
#   2) POS distribution (all songs)
#   3) Top tokens (all songs, excluding SPACE/PUNCT)
#   4) Repeated phrases (sentence-level)
#   5) Top emotional density sentences (min_words + dedup)
#   6) Saves sentence-level density results to emotional_density_sentences.csv

# Emotional density (sentence-level):
#   density = (# interjections + # contractions + # slang tokens) / (# word tokens)
#   - Word tokens exclude punctuation and spaces
#   - This is an interpretable “expression density” metric (not emotion classification)

In [2]:
import os
import re
import csv
from dataclasses import dataclass
from typing import Dict, List, Tuple
from collections import Counter, defaultdict

import spacy


In [3]:

# ===============================
# 1) Configuration
# ===============================

LYRICS_DIR = "./lyrics"
EXAMPLE_FILE = "song_001.txt"
SPACY_MODEL = "en_core_web_md"  # md/lg recommended for meaningful token.is_oov

# Emotional / informal marker lexicons (expandable)
INTJ_SET = {
    "oh", "yeah", "hey", "hmm", "ooh", "wow", "ah", "uh", "uhh",
    "please", "sorry", "damn", "hello", "ayy", "yuh"
}

CONTRACTION_SUFFIXES = {"'d", "'ll", "'m", "'re", "'s", "'ve", "n't"}

SLANG_SET = {
    "wanna", "gonna", "gotta", "ain't", "cuz", "ya", "bout",
    "nothin", "lovin", "fuckin", "gon", "tryna", "kinda"
}

# Simple normalization helpers
QUOTE_MAP = str.maketrans({"’": "'", "‘": "'", "´": "'", "`": "'"})
DASH_MAP = str.maketrans({"—": "-", "–": "-"})
DROP_G_PATTERN = re.compile(r"^([a-z]+)in'$", re.IGNORECASE)


# ===============================
# 2) Utilities (normalization + printing)
# ===============================

def normalize_text(s: str) -> str:
    """Unify quotes/dashes and trim whitespace."""
    return s.translate(QUOTE_MAP).translate(DASH_MAP).strip()

def normalize_token(t: str) -> str:
    """Normalize token for stable counting (lowercase + quote/dash + optional drop-g handling)."""
    t = normalize_text(t).lower()
    m = DROP_G_PATTERN.match(t)
    if m:
        t = m.group(1) + "in"
    return t

def section(title: str) -> None:
    """Pretty section header for console output."""
    print("=" * 70)
    print(title)
    print("=" * 70)

def format_evidence(evidence: Dict[str, List[str]]) -> str:
    """Format evidence tokens for readability."""
    if not evidence:
        return "-"
    order = ["INTJ", "CONTRACTION", "SLANG"]
    parts = []
    for cat in order:
        if cat in evidence and evidence[cat]:
            toks = evidence[cat][:6]  # avoid overly long lines
            parts.append(f"{cat}: {', '.join(toks)}")
    return " | ".join(parts) if parts else "-"

def print_counter(title: str, counter: Counter, k: int = 10) -> None:
    section(title)
    for i, (key, val) in enumerate(counter.most_common(k), 1):
        print(f"{i:>2}. {key:<12} {val}")
    print()

def print_repeated_phrases(phrases: Counter, k: int = 10, min_count: int = 2) -> None:
    section(f"Repeated Phrases (count ≥ {min_count})")
    shown = 0
    for phrase, cnt in phrases.most_common():
        if cnt >= min_count:
            shown += 1
            print(f"{shown:>2}. ({cnt}x) {phrase}")
            if shown >= k:
                break
    if shown == 0:
        print("No repeated phrases found.")
    print()

def print_emotional_top(
    rows: List["EmotionalDensityRow"],
    k: int = 5,
    min_words: int = 4,
    dedup: bool = True
) -> None:
    """
    Print top-k sentences by emotional density.
    - min_words filters out very short lines (e.g., "Yeah")
    - dedup removes duplicate sentences for cleaner summaries
    """
    section(f"Top Emotional Density Sentences (Top {k}, min_words={min_words}, dedup={dedup})")

    filtered = [r for r in rows if r.word_total >= min_words]
    if not filtered:
        print("No sentences meet the minimum word length.\n")
        return

    if dedup:
        best_by_sentence: Dict[str, EmotionalDensityRow] = {}
        for r in filtered:
            key = r.sentence.lower()
            if (key not in best_by_sentence) or (r.density > best_by_sentence[key].density):
                best_by_sentence[key] = r
        filtered = list(best_by_sentence.values())

    top = sorted(filtered, key=lambda x: x.density, reverse=True)[:k]
    for i, r in enumerate(top, 1):
        print(f"{i:>2}. density={r.density:.2f}  ({r.emo_total}/{r.word_total})")
        print(f"    {r.sentence}")
        print(f"    breakdown={r.breakdown} | evidence={format_evidence(r.evidence_tokens)}")
    print()

def print_error_summary(errors: List["ErrorToken"], k_words: int = 12) -> None:
    """Readable summary of error candidates (OOV or POS='X'), excluding punctuation/spaces."""
    section("Error Candidates (Readable Summary)")
    if not errors:
        print("No error candidates found.\n")
        return

    word_freq = Counter([e.word for e in errors])
    pos_freq = Counter([e.pos for e in errors])
    oov_count = sum(1 for e in errors if e.oov)

    print(f"- total error tokens: {len(errors)}")
    print(f"- OOV tokens: {oov_count} ({(oov_count/len(errors))*100:.1f}%)\n")

    print("Top words:")
    for i, (w, c) in enumerate(word_freq.most_common(k_words), 1):
        print(f" {i:>2}. {w:<12} {c}")
    print()

    print("POS distribution among error candidates:")
    for i, (p, c) in enumerate(pos_freq.most_common(10), 1):
        print(f" {i:>2}. {p:<8} {c}")
    print()

    print("Sample error candidates (first 15):")
    for e in errors[:15]:
        print(f" - '{e.word}' (norm='{normalize_token(e.word)}') | POS={e.pos} | OOV={e.oov}")
    print()


# ===============================
# 3) Data structures
# ===============================

@dataclass
class EmotionalDensityRow:
    file: str
    sentence: str
    density: float
    emo_total: int
    word_total: int
    breakdown: Dict[str, int]
    evidence_tokens: Dict[str, List[str]]

@dataclass
class ErrorToken:
    file: str
    word: str
    pos: str
    oov: bool


# ===============================
# 4) Loading / saving
# ===============================

def load_lyrics_files(folder: str) -> Dict[str, str]:
    """Load all .txt lyric files from a directory."""
    lyrics = {}
    for fname in os.listdir(folder):
        if fname.endswith(".txt"):
            with open(os.path.join(folder, fname), encoding="utf-8") as f:
                lyrics[fname] = f.read()
    return lyrics

def unique_errors(errors: List[ErrorToken]) -> List[ErrorToken]:
    """Deduplicate error tokens per file (file, word)."""
    seen = set()
    out = []
    for e in errors:
        key = (e.file, e.word)
        if key not in seen:
            seen.add(key)
            out.append(e)
    return out

def save_density_csv(path: str, rows: List[EmotionalDensityRow]) -> None:
    """Save sentence-level emotional density results to CSV (for later analysis)."""
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["file", "sentence", "density", "emo_total", "word_total", "breakdown", "evidence"])
        for r in rows:
            w.writerow([
                r.file, r.sentence, f"{r.density:.6f}",
                r.emo_total, r.word_total,
                str(r.breakdown), str(r.evidence_tokens)
            ])


# ===============================
# 5) Analyzer
# ===============================

class LyricsAnalyzer:
    def __init__(self, nlp):
        self.nlp = nlp

    def emotional_category(self, tok) -> Tuple[bool, str]:
        """Return (True, category) if token matches an emotional/informal marker."""
        t = normalize_token(tok.text)
        if t in INTJ_SET:
            return True, "INTJ"
        if t in SLANG_SET:
            return True, "SLANG"
        if t in CONTRACTION_SUFFIXES:
            return True, "CONTRACTION"
        return False, "NONE"

    def analyze_song(self, text: str, fname: str):
        """Analyze one song: POS, token frequency, repetition, density, and OOV error candidates."""
        doc = self.nlp(normalize_text(text))

        pos_counts = Counter()
        token_counts = Counter()
        phrase_counts = Counter()
        density_rows: List[EmotionalDensityRow] = []
        errors: List[ErrorToken] = []

        # Sentence-level: repetition + emotional density
        for sent in doc.sents:
            sent_text = normalize_text(sent.text)
            phrase_counts[sent_text.lower()] += 1

            word_total = 0
            emo_total = 0
            breakdown = Counter()
            evidence = defaultdict(list)

            for tok in sent:
                if tok.is_space or tok.is_punct:
                    continue
                word_total += 1

                is_emo, cat = self.emotional_category(tok)
                if is_emo:
                    emo_total += 1
                    breakdown[cat] += 1
                    evidence[cat].append(tok.text)

            density_rows.append(
                EmotionalDensityRow(
                    file=fname,
                    sentence=sent_text,
                    density=(emo_total / word_total) if word_total else 0.0,
                    emo_total=emo_total,
                    word_total=word_total,
                    breakdown=dict(breakdown),
                    evidence_tokens=dict(evidence),
                )
            )

        # Document-level: POS + top tokens + error candidates
        for tok in doc:
            pos_counts[tok.pos_] += 1

            if not (tok.is_space or tok.is_punct):
                token_counts[normalize_token(tok.text)] += 1

            if not (tok.is_space or tok.is_punct) and (tok.is_oov or tok.pos_ == "X"):
                errors.append(ErrorToken(fname, tok.text, tok.pos_, tok.is_oov))

        return pos_counts, token_counts, phrase_counts, density_rows, errors


# ===============================
# 6) Main
# ===============================

def main() -> None:
    nlp = spacy.load(SPACY_MODEL)
    analyzer = LyricsAnalyzer(nlp)

    lyrics_data = load_lyrics_files(LYRICS_DIR)

    section("LOAD")
    print(f"Loaded {len(lyrics_data)} lyrics files from: {LYRICS_DIR}\n")

    agg_pos = Counter()
    agg_tokens = Counter()
    agg_phrases = Counter()
    all_density: List[EmotionalDensityRow] = []
    all_errors: List[ErrorToken] = []

    # Analyze all songs
    for fname, text in lyrics_data.items():
        pos_counts, token_counts, phrase_counts, density_rows, errors = analyzer.analyze_song(text, fname)

        agg_pos.update(pos_counts)
        agg_tokens.update(token_counts)
        agg_phrases.update(phrase_counts)
        all_density.extend(density_rows)
        all_errors.extend(errors)

    # Dataset-level outputs
    print_counter("POS Distribution (All Songs)", agg_pos, k=15)
    print_counter("Top Tokens (All Songs)", agg_tokens, k=20)
    print_repeated_phrases(agg_phrases, k=15, min_count=2)
    print_emotional_top(all_density, k=5, min_words=4, dedup=True)

    # Example song outputs
    if EXAMPLE_FILE in lyrics_data:
        section(f"EXAMPLE: {EXAMPLE_FILE}")
        pos_counts, token_counts, phrase_counts, density_rows, errors = analyzer.analyze_song(
            lyrics_data[EXAMPLE_FILE], EXAMPLE_FILE
        )

        print_counter("POS Distribution (Example)", pos_counts, k=20)
        print_counter("Top Tokens (Example)", token_counts, k=12)
        print_repeated_phrases(phrase_counts, k=10, min_count=2)
        print_emotional_top(density_rows, k=5, min_words=4, dedup=True)

        errors_unique = unique_errors(errors)
        print_error_summary(errors_unique, k_words=12)

    # Save CSV
    save_density_csv("emotional_density_sentences.csv", all_density)
    section("SAVE")
    print("Saved: emotional_density_sentences.csv\n")


if __name__ == "__main__":
    main()


LOAD
Loaded 100 lyrics files from: ./lyrics

POS Distribution (All Songs)
 1. PRON         9009
 2. VERB         6281
 3. SPACE        5664
 4. NOUN         4848
 5. AUX          3639
 6. PUNCT        3017
 7. ADP          2883
 8. ADV          2411
 9. DET          2152
10. ADJ          1717
11. PART         1441
12. SCONJ        1200
13. PROPN        1094
14. CCONJ        1052
15. INTJ         1050

Top Tokens (All Songs)
 1. i            2523
 2. you          2016
 3. the          1001
 4. it           779
 5. me           751
 6. and          713
 7. n't          693
 8. my           630
 9. to           628
10. 'm           591
11. a            514
12. do           500
13. that         474
14. we           421
15. in           418
16. your         399
17. on           389
18. 's           382
19. like         331
20. oh           324

Repeated Phrases (count ≥ 2)
 1. (49x) yeah
 2. (38x) yeah,
 3. (12x) it's you babe
 4. (12x) and i'm a sucker
for the way that you move
babe
 5. (1